In [1]:
# prompt: connect to drive

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
!mv /content/drive/MyDrive/YelpReviews_Model.rar /content/

In [3]:
!unrar x /content/YelpReviews_Model.rar



UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/YelpReviews_Model.rar

Creating    NLPModelTrained                                           OK
Extracting  NLPModelTrained/NLPModel.h5                                    7% 14% 21% 28% 35% 42% 49% 56% 63% 70% 78% 85% 92% 95%  OK 
Extracting  NLPModelTrained/tokenizer.pickle                              99%  OK 
All OK


In [4]:
from tensorflow.keras.models import load_model
import pickle

# Load the model
model = load_model('NLPModelTrained/NLPModel.h5')

# Load the tokenizer
with open('NLPModelTrained/tokenizer.pickle', 'rb') as f:
    tokenizer = pickle.load(f)

In [5]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.regularizers import l2
import tensorflow_datasets as tfds
import numpy as np

# Load Yelp Polarity dataset
dataset, info = tfds.load('yelp_polarity_reviews', with_info=True, as_supervised=True)
train_data, test_data = dataset['train'], dataset['test']

# Prepare the data
vocab_size = 50000  # Limit vocabulary to top 50,000 words
max_length = 200    # Maximum review length (truncation/padding)


# Convert text data to sequences and pad them
def preprocess_dataset(dataset):
    texts, labels = [], []
    for text, label in dataset:
        texts.append(text.numpy().decode('utf-8'))
        labels.append(label.numpy())
    sequences = tokenizer.texts_to_sequences(texts)
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
    return padded_sequences, np.array(labels)

# Preprocess testing data
test_x, test_y = preprocess_dataset(test_data)


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/yelp_polarity_reviews/incomplete.7WQSSN_0.2.0/yelp_polarity_reviews-train.…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/yelp_polarity_reviews/incomplete.7WQSSN_0.2.0/yelp_polarity_reviews-test.t…

Dataset yelp_polarity_reviews downloaded and prepared to /root/tensorflow_datasets/yelp_polarity_reviews/0.2.0. Subsequent calls will reuse this data.


In [6]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# Function to predict sentiment of input text
def predict_sentiment(model, text, tokenizer, max_length=200):
    # Convert text to sequence using the tokenizer
    sequence = tokenizer.texts_to_sequences([text])

    # Pad the sequence to the fixed length
    padded_sequence = pad_sequences(sequence, maxlen=max_length, padding='post', truncating='post')

    # Get prediction probability
    prediction = model.predict(padded_sequence)[0][0]

    # Classify sentiment
    sentiment = "Positive" if prediction > 0.5 else "Negative"

    # Confidence score
    confidence = prediction if sentiment == "Positive" else 1 - prediction

    return sentiment, confidence

In [7]:
print(len(test_x))

38000


In [8]:
# Function to decode a sequence back to text
def sequence_to_text(sequence, tokenizer):
    return tokenizer.sequences_to_texts([sequence])[0]

# Select random samples from the test set
num_samples = 5  # Number of samples to test
random_indices = np.random.choice(len(test_x), num_samples, replace=False)

# Test the selected samples
for i in random_indices:
    # Get the sequence and label
    sequence = test_x[i]
    true_label = test_y[i]

    # Decode the sequence to text
    review_text = sequence_to_text(sequence, tokenizer)
    print(review_text)
    print("")


    # Predict sentiment using the model
    predicted_sentiment, confidence = predict_sentiment(model, review_text, tokenizer)

    # Map the true label to a sentiment
    true_sentiment = "Positive" if true_label == 1 else "Negative"

    # Print results
    print(f"Review: {review_text}")
    print(f"True Sentiment: {true_sentiment}")
    print(f"Predicted Sentiment: {predicted_sentiment} (Confidence: {confidence:.4f})")
    print("-" * 50)


always delicious really nice people as well only complaint is that the delivery takes a little while if they delivered faster they would be the best

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Review: always delicious really nice people as well only complaint is that the delivery takes a little while if they delivered faster they would be the best
True Sentiment: Positive
Predicted Sentiment: Positive (Confidence: 0.9966)
--------------------------------------------------
matt is very attentive he has lots of patience and seems to be more into the customer service side of things than sales i ended up purchasing more product because he was so helpful cool cat for sure their product seems to taste less synthetic than other e juice i've purchased from other stores

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
Review: matt is very attentive he has lots of patience and seems to be more into the customer service side of things than sales i ended up purchasing more product because he was so helpful cool ca

In [44]:
# Evaluate the model
test_loss, test_acc = model.evaluate(test_x, test_y)
print(f"Test Accuracy: {test_acc:.4f}")

1188/1188 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - accuracy: 0.9469 - loss: 0.1579
Test Accuracy: 0.9472


In [60]:
import tensorflow as tf
import numpy as np
from typing import List, Tuple
import random
import string
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
class DeepWordBug:
    def __init__(self, model_path: str, tokenizer, max_length: int = 200, scaling_factor: float = 0.1, max_epsilon: int = 30, strategy: str = "proportional", lambda_param: float = 0.5):
        """
        Initialize DeepWordBug attack for TensorFlow model.

        Args:
            model_path: Path to saved Keras model
            tokenizer: Keras tokenizer used for the model
            max_length: Maximum sequence length for padding
            scaling_factor: A factor to scale epsilon relative to text length
            max_epsilon: The maximum allowed epsilon value
            strategy: Strategy for calculating epsilon ("proportional" or "fixed")
            lambda_param: Weight for combining temporal scoring function
        """
        self.model = keras.models.load_model(model_path)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.scaling_factor = scaling_factor
        self.max_epsilon = max_epsilon
        self.strategy = strategy
        self.lambda_param = lambda_param

    def calculate_epsilon(self, text: str) -> int:
        """
        Calculates epsilon (maximum edit distance) for DeepWordBug based on text length and a scaling strategy.

        Args:
            text (str): The input text.

        Returns:
            int: The calculated epsilon value.
        """
        words = text.split()
        num_words = len(words)

        if self.strategy == "proportional":
            epsilon = int(num_words * self.scaling_factor)  # Epsilon proportional to text length
            epsilon = min(epsilon, self.max_epsilon)  # Cap epsilon at max_epsilon
        elif self.strategy == "fixed":
            epsilon = self.max_epsilon  # Fixed epsilon value
        else:
            raise ValueError("Invalid epsilon calculation strategy.")

        return epsilon

    def predict(self, text: str) -> float:
        """Get model prediction for a text input."""
        sequence = self.tokenizer.texts_to_sequences([text])
        padded = pad_sequences(sequence, maxlen=self.max_length, padding='post', truncating='post')
        prediction = self.model.predict(padded, verbose=0)[0][0]
        return prediction

    def score_r1s(self, text: str, word_position: int) -> float:
        """Calculate Replace-1 Score for a word."""
        words = text.split()
        original_score = self.predict(text)

        if 0 <= word_position < len(words):
            modified_words = words.copy()
            modified_words[word_position] = '<UNK>'
            modified_text = ' '.join(modified_words)
            modified_score = self.predict(modified_text)
            return abs(original_score - modified_score)
        return 0.0

    def score_ths(self, text: str, word_position: int) -> float:
        """Calculate Temporal Head Score."""
        words = text.split()
        if 0 <= word_position < len(words):
            head_text = ' '.join(words[:word_position + 1])
            partial_text = ' '.join(words[:word_position])
            return abs(self.predict(head_text) - self.predict(partial_text))
        return 0.0

    def score_tts(self, text: str, word_position: int) -> float:
        """Calculate Temporal Tail Score."""
        words = text.split()
        if 0 <= word_position < len(words):
            tail_text = ' '.join(words[word_position:])
            partial_text = ' '.join(words[word_position + 1:])
            return abs(self.predict(tail_text) - self.predict(partial_text))
        return 0.0

    def score_combined(self, text: str, word_position: int) -> float:
        """Calculate Combined Score using THS and TTS."""
        ths = self.score_ths(text, word_position)
        tts = self.score_tts(text, word_position)
        return ths + self.lambda_param * tts

    def transform_token(self, token: str, method: str = None) -> str:
        """Transform a token using various methods."""
        if len(token) <= 1:
            return token

        methods = ['swap', 'substitute', 'delete', 'insert']
        method = method or random.choice(methods)

        if method == 'swap':
            pos = random.randint(0, len(token) - 2)
            chars = list(token)
            chars[pos], chars[pos + 1] = chars[pos + 1], chars[pos]
            return ''.join(chars)

        elif method == 'substitute':
            pos = random.randint(0, len(token) - 1)
            chars = list(token)
            chars[pos] = random.choice(string.ascii_lowercase)
            return ''.join(chars)

        elif method == 'delete':
            pos = random.randint(0, len(token) - 1)
            return token[:pos] + token[pos + 1:]

        elif method == 'insert':
            pos = random.randint(0, len(token))
            char = random.choice(string.ascii_lowercase)
            return token[:pos] + char + token[pos:]

    def calculate_edit_distance(self, s1: str, s2: str) -> int:
        """Calculate Levenshtein distance between two strings."""
        if len(s1) < len(s2):
            return self.calculate_edit_distance(s2, s1)

        if len(s2) == 0:
            return len(s1)

        previous_row = range(len(s2) + 1)
        for i, c1 in enumerate(s1):
            current_row = [i + 1]
            for j, c2 in enumerate(s2):
                insertions = previous_row[j + 1] + 1
                deletions = current_row[j] + 1
                substitutions = previous_row[j] + (c1 != c2)
                current_row.append(min(insertions, deletions, substitutions))
            previous_row = current_row

        return previous_row[-1]

    def attack(self, text: str, scoring_method: str = 'r1s') -> Tuple[str, float, float]:
        """
        Generate adversarial text using DeepWordBug attack.

        Args:
            text: Input text to attack
            scoring_method: Method to score tokens ('r1s', 'ths', 'tts', 'combined')

        Returns:
            Tuple of (adversarial_text, original_score, adversarial_score)
        """
        words = text.split()

        # Calculate epsilon based on the text length
        epsilon = self.calculate_epsilon(text)

        # Score each word
        scores = []
        for i, word in enumerate(words):
            if scoring_method == 'r1s':
                score = self.score_r1s(text, i)
            elif scoring_method == 'ths':
                score = self.score_ths(text, i)
            elif scoring_method == 'tts':
                score = self.score_tts(text, i)
            else:  # combined
                score = self.score_combined(text, i)
            scores.append((i, score))

        # Sort words by importance score
        sorted_words = sorted(scores, key=lambda x: x[1], reverse=True)

        # Modify words while respecting the total edit distance limit
        modified_words = words.copy()
        total_edit_distance = 0

        for idx, _ in sorted_words:
            original_word = words[idx]
            modified_word = self.transform_token(original_word)

            # Calculate edit distance for this modification
            edit_dist = self.calculate_edit_distance(original_word, modified_word)

            # Check if the total edit distance exceeds epsilon
            if total_edit_distance + edit_dist > epsilon:
                continue  # Skip this modification to stay within the limit

            # Apply the modification
            modified_words[idx] = modified_word
            total_edit_distance += edit_dist

        adversarial_text = ' '.join(modified_words)
        return adversarial_text

# Example usage
if __name__ == "__main__":
    attack = DeepWordBug(
        model_path='NLPModelTrained/NLPModel.h5',
        tokenizer=tokenizer,  # My saved tokenizer
        max_length=200,
        scaling_factor=0.2,
        max_epsilon=5,
        strategy="proportional"
    )



In [61]:
from tqdm import tqdm

def apply_attack_to_dataset(attack: DeepWordBug, test_x: List[str], scoring_method: str = 'r1s') -> Tuple[List[str], List[float], List[float], List[int]]:
    """
    Apply DeepWordBug attack to the entire dataset with progress tracking.

    Args:
        attack: DeepWordBug attack instance
        test_x: List of input texts
        test_y: List of true labels
        scoring_method: Scoring method for the attack ('r1s', 'ths', 'tts', 'combined')

    Returns:
        Tuple of (adversarial_texts, original_scores, adversarial_scores, true_labels)
    """
    adversarial_texts = []

    # Iterate over the dataset with progress bar
    for text in tqdm(test_x, total=len(test_x), desc="Applying DeepWordBug attack"):
        try:
            adv_text = attack.attack(text, scoring_method=scoring_method)
            # Store results
            adversarial_texts.append(adv_text)

        except Exception as e:
            continue

    return adversarial_texts




In [63]:
sequences = []
for text in test_x[:100]:
    sec = sequence_to_text(text, tokenizer)
    sequences.append(sec)

print(len(sequences))


100


In [64]:
# Apply attack to the entire dataset
adv_texts= apply_attack_to_dataset(
attack=attack,
test_x=sequences,
)


Applying DeepWordBug attack: 100%|██████████| 100/100 [30:48<00:00, 18.48s/it]


In [66]:
# prompt: compare between the model acc on adv_texts and sequences

import numpy as np

# Assuming 'model', 'tokenizer', 'test_x', and 'test_y' are defined from the previous code

# Function to predict sentiment of input text (already defined)
# ... (rest of the code from the previous response)


def evaluate_model_on_texts(model, texts, labels, tokenizer):
    """Evaluates the model on a list of texts and returns the accuracy."""
    sequences = tokenizer.texts_to_sequences(texts)
    padded_sequences = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')
    _, accuracy = model.evaluate(padded_sequences, labels, verbose=0)
    return accuracy


# Evaluate the model on adversarial texts
adv_labels = test_y[:len(adv_texts)] # Assuming adv_texts corresponds to the first part of the test set.
adv_acc = evaluate_model_on_texts(model, adv_texts, adv_labels, tokenizer)
print(f"Accuracy on Adversarial Texts: {adv_acc:.4f}")

# Evaluate the model on original sequences
original_sequences = [sequence_to_text(seq, tokenizer) for seq in test_x[:len(adv_texts)]]
orig_labels = test_y[:len(adv_texts)]
orig_acc = evaluate_model_on_texts(model, original_sequences, orig_labels, tokenizer)
print(f"Accuracy on Original Sequences: {orig_acc:.4f}")

print(f"Difference in accuracy: {orig_acc - adv_acc:.4f}")


Accuracy on Adversarial Texts: 0.8000
Accuracy on Original Sequences: 0.9700
Difference in accuracy: 0.1700


In [105]:

misclassified_indices = []
for i in range(len(adv_texts)):
    original_text = original_sequences[i]
    adv_text = adv_texts[i]
    true_label = orig_labels[i]

    orig_sentiment, _ = predict_sentiment(model, original_text, tokenizer)
    adv_sentiment, _ = predict_sentiment(model, adv_text, tokenizer)

    true_sentiment = "Positive" if true_label == 1 else "Negative"

    if orig_sentiment != adv_sentiment and orig_sentiment == true_sentiment:
        misclassified_indices.append(i)


num_to_print = 5  # Number of misclassified examples to print
printed_count = 0

for i in misclassified_indices:
    if printed_count >= num_to_print:
        break

    original_text = original_sequences[i]
    adv_text = adv_texts[i]
    true_label = orig_labels[i]
    true_sentiment = "Positive" if true_label == 1 else "Negative"
    orig_sentiment, _ = predict_sentiment(model, original_text, tokenizer)
    adv_sentiment, _ = predict_sentiment(model, adv_text, tokenizer)

    print(f"Original Text:\n{original_text}\n")
    print(f"Adversarial Text:\n{adv_text}\n")
    print(f"True Sentiment: {true_sentiment}")
    print(f"Original Predicted Sentiment: {orig_sentiment}")
    print(f"Adversarial Predicted Sentiment: {adv_sentiment}\n")
    print("-" * 50)

    printed_count += 1


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
1/1 ━━━━━━━━

In [81]:
def printing_results(text, adv_text):
    # Predict sentiment of the original text
    sentiment_orig, confidence_orig = predict_sentiment(model, text, tokenizer)

    # Predict sentiment of the adversarial text
    sentiment_adversarial, confidence_adversarial = predict_sentiment(model, adv_text, tokenizer)

    # Print original results
    print(f"Original Sentiment: {sentiment_orig} (Confidence: {confidence_orig:.4f})")
    print("-" * 50)

    # Print adversarial results
    print(f"Adversarial Sentiment: {sentiment_adversarial} (Confidence: {confidence_adversarial:.4f})")
    print("-" * 50)

    # Print original and adversarial text/predictions
    print(f"Original text: {text}")
    print(f"Adversarial text: {adv_text}")
    print("-" * 50)

    # Check if the attack succeeded
    if sentiment_orig != sentiment_adversarial:
        print("Attack SUCCEEDED: Sentiment flipped!")
    else:
        print("Attack FAILED: Sentiment did not flip.")

In [84]:
text = "This restaurant was excellent and I really enjoyed the service"
adv_text = attack.attack(text)

printing_results(text,adv_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Original Sentiment: Positive (Confidence: 0.9946)
--------------------------------------------------
Adversarial Sentiment: Negative (Confidence: 0.7077)
--------------------------------------------------
Original text: This restaurant was excellent and I really enjoyed the service
Adversarial text: This restaurant was xcellent and I really enjoyjed the service
--------------------------------------------------
Attack SUCCEEDED: Sentiment flipped!


In [92]:
text2="This food was so bad"
adv_text2= attack.attack(text2)
printing_results(text2,adv_text2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Original Sentiment: Negative (Confidence: 0.9929)
--------------------------------------------------
Adversarial Sentiment: Negative (Confidence: 0.6642)
--------------------------------------------------
Original text: This food was so bad
Adversarial text: This food was so ba
--------------------------------------------------
Attack FAILED: Sentiment did not flip.


In [93]:
text3 = "I loved the food so much"
adv_text3 = attack.attack(text3)
printing_results(text3,adv_text3)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
Original Sentiment: Positive (Confidence: 0.8359)
--------------------------------------------------
Adversarial Sentiment: Negative (Confidence: 0.8747)
--------------------------------------------------
Original text: I loved the food so much
Adversarial text: I floved the food so much
--------------------------------------------------
Attack SUCCEEDED: Sentiment flipped!


In [94]:
text = "This restaurant was excellent and I really enjoyed the service"
adv_text = attack.attack(text)

printing_results(text,adv_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Original Sentiment: Positive (Confidence: 0.9946)
--------------------------------------------------
Adversarial Sentiment: Negative (Confidence: 0.7077)
--------------------------------------------------
Original text: This restaurant was excellent and I really enjoyed the service
Adversarial text: This restaurant was exczllent and I really enjmyed the service
--------------------------------------------------
Attack SUCCEEDED: Sentiment flipped!
